![Pythonista](img/pythonista.png)

# Matriz avanzada de versiones y promoción por entornos.

Este capítulo asume la base vista previmante y se centra en decisiones operativas: cómo diseñar una `strategy.matrix` útil, cómo controlar costo/tiempo de ejecución y cómo promover el mismo artefacto entre entornos sin reconstruir.

## Objetivos.

- Diseñar matrices con dimensiones relevantes y evitar combinaciones inútiles.
- Aplicar `include`, `exclude`, `fail-fast` y `max-parallel` con criterio de costo-riesgo.
- Relacionar *runners* efímeros con artefactos para promoción trazable.
- Conectar validación por matriz con despliegue controlado por entorno.

## Modelo operativo: matriz, artefactos y entornos.

En un *workflow* real, estas tres piezas se diseñan juntas:

- **matriz**: decide en qué combinaciones validar (por ejemplo versión de *Python* y sistema operativo);
- **artefactos**: preservan resultados del *build* en *runners* efímeros;
- **entornos**: gobiernan la promoción con aprobaciones, secretos y políticas.

La regla práctica es esta: validar en paralelo, construir una sola vez cuando corresponda, y promover exactamente el mismo artefacto entre entornos con controles crecientes.

Este enfoque evita dos fallas comunes: sobre-ejecutar combinaciones de poco valor y desplegar binarios distintos a los que pasaron validación.

## Qué agrega este capítulo.

Previamente se cubrió la sintaxis base de `jobs`, `needs`, `if` y `environment`. Aquí el foco cambia a decisiones de diseño:

- cuándo una dimensión debe entrar a la matriz y cuándo no;
- cómo recortar combinaciones con `exclude` sin perder cobertura útil;
- cuándo forzar `fail-fast: true` y cuándo conviene desactivarlo;
- cómo usar `max-parallel` para limitar costo de ejecución.

También se refuerza una idea clave: los nombres de entorno (`dev`, `qa`, `staging`, `prod`) son patrones comunes, no un estándar universal. Cada organización define su esquema según su contexto operativo y regulatorio.

## Patrones avanzados de `strategy.matrix`.

```yaml
jobs:
  tests:
    runs-on: ${{ matrix.os }}
    strategy:
      fail-fast: true
      max-parallel: 4
      matrix:
        os: [ubuntu-latest, windows-latest]
        python-version: ["3.10", "3.11", "3.12"]
        include:
          - os: ubuntu-latest
            python-version: "3.13"   # caso extra controlado
        exclude:
          - os: windows-latest
            python-version: "3.10"   # combinación de bajo valor
    steps:
      - uses: actions/checkout@v6
      - uses: actions/setup-python@v6
        with:
          python-version: ${{ matrix.python-version }}
      - run: pytest
```

Este patrón permite cubrir compatibilidad real sin disparar una explosión combinatoria. La matriz deja de ser “todas las combinaciones posibles” y pasa a ser “las combinaciones que reducen riesgo de forma costo-efectiva”.

## Promoción por entornos sin reconstrucción.

El flujo recomendado es: validar por matriz, construir artefacto una vez, y promover ese mismo artefacto a `test` y `prod` con controles distintos.

```yaml
jobs:
  tests:
    strategy:
      matrix:
        python-version: ["3.11", "3.12"]
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v6
      - run: pytest

  build:
    runs-on: ubuntu-latest
    needs: tests
    steps:
      - uses: actions/checkout@v6
      - run: python -m build
      - uses: actions/upload-artifact@v7
        with:
          name: paquete
          path: dist/

  deploy_test:
    runs-on: ubuntu-latest
    needs: build
    environment: test
    steps:
      - uses: actions/download-artifact@v8
        with:
          name: paquete
          path: dist/
      - run: echo "despliegue en test"

  deploy_prod:
    runs-on: ubuntu-latest
    needs: deploy_test
    environment: prod
    steps:
      - uses: actions/download-artifact@v8
        with:
          name: paquete
          path: dist/
      - run: echo "despliegue en prod"
```

Así se evita probar una cosa y desplegar otra: cambia el nivel de control del entorno, no el binario que se promueve.

## Nota de seguridad: riesgo de *cache poisoning*.

Cuando se usa caché de dependencias (por ejemplo con *actions/cache*), una clave demasiado amplia o reutilizada entre contextos de confianza distinta puede permitir contaminación de caché.

Buenas prácticas mínimas:

- separar claves por sistema operativo, versión de *Python* y tipo de evento;
- evitar restaurar caché de *pull requests* no confiables hacia flujos de despliegue;
- no usar `restore-keys` demasiado genéricas en *jobs* sensibles;
- fijar acciones por versión mayor estable y, en flujos críticos, preferir *pinning* por SHA.

La idea es la misma que con artefactos: la aceleración del pipeline no debe degradar la integridad de lo que se ejecuta o despliega.

In [1]:
from itertools import product

python_versions = ["3.10", "3.11", "3.12"]
oses = ["ubuntu-latest", "windows-latest"]

# Matriz base (producto cartesiano).
combinaciones = list(product(python_versions, oses))

# Exclusion de combinaciones de bajo valor para ahorrar costo/tiempo.
excluir = {("3.10", "windows-latest")}
matriz_final = [c for c in combinaciones if c not in excluir]

print("Combinaciones totales:", len(combinaciones))
print("Combinaciones finales:", len(matriz_final))
print("Detalle:")
for py, os_name in matriz_final:
    print(f"- python={py}, os={os_name}")

Combinaciones totales: 6
Combinaciones finales: 5
Detalle:
- python=3.10, os=ubuntu-latest
- python=3.11, os=ubuntu-latest
- python=3.11, os=windows-latest
- python=3.12, os=ubuntu-latest
- python=3.12, os=windows-latest


<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>